In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from bk_tools import BASE_DIR
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from pathlib import Path
# Import all necessary libraries for data manipulation, visualization and deep learning.

In [ ]:
import tensorflow as tf
print("GPUs:", tf.config.list_physical_devices('GPU'))

# Enabling GPU if available

In [ ]:
BASE_DIR = Path(__file__).resolve().parent
path_db = BASE_DIR / "BreaKHis_v1" / "histology_slides" / "breast"


def packup_details(f):
    p = Path(f)

    # only filename, cross-platform safe
    filename = p.name                          # SOB_B_A-14-22549G-100-001.png
    stem = p.stem                             # SOB_B_A-14-22549G-100-001

    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected filename format: {filename}")

    example = parts[0]                        # SOB
    is_malign = 1 if parts[1] == "M" else 0   # M -> malignant, B -> benign

    names = parts[2].split("-")
    if len(names) < 5:
        raise ValueError(f"Unexpected tumor name format: {filename}")

    class_tumor = names[0]
    year = int(names[1]) + 2000
    patient_id = names[2]
    zoom = int(names[3])
    file_id = names[4]

    return {
        "patient_id": patient_id,
        "file_id": file_id,
        "example": example,
        "class": class_tumor,
        "year": year,
        "zoom": zoom,
        "file_path": str(p),
        "is_malign": is_malign,
    }



def prepare_data_table(rootpath=path_db) -> pd.DataFrame:
    rootpath = Path(rootpath)

    # cross-platform recursive PNG search
    files = list(rootpath.rglob("*.png"))

    if not files:
        raise FileNotFoundError(f"No PNG files found under: {rootpath}")

    datas = []
    bad_files = []

    for f in files:
        try:
            datas.append(packup_details(f))
        except Exception as e:
            bad_files.append((str(f), str(e)))

    df = pd.DataFrame(datas)

    print("DataFrame shape:", df.shape)
    print("DataFrame columns:", df.columns.tolist())
    print("Parsed files:", len(datas))
    print("Failed files:", len(bad_files))

    if bad_files:
        print("\nFirst 10 problematic files:")
        for fp, err in bad_files[:10]:
            print(fp, "->", err)

    return df


In [ ]:
# Prepare the data table using the custom function from bk_tools and display its information.
df = prepare_data_table()
df.info()
df["class"].unique()

In [ ]:
# Patient-wise split with class coverage in all splits (train/val/test).
chosen_zoom = 40
test_val_size = 0.3
seed = 42

# Filtering with selected zoom
df_zoom = df[df["zoom"] == chosen_zoom].copy()
print(f"Selected zoom: {chosen_zoom}")
print(f"Total samples at selected zoom: {len(df_zoom)}")
print(f"Total unique patients at selected zoom: {df_zoom['patient_id'].nunique()}")

print("\nInitial image-level class distribution:")
print(df_zoom["class"].value_counts())

# Remove patients that contain multiple classes to avoid leakage/noisy labels.
patient_class_counts = df_zoom.groupby("patient_id")["class"].nunique()
problematic_patients = patient_class_counts[patient_class_counts > 1].index.tolist()
if len(problematic_patients) > 0:
    print(f"\nRemoving {len(problematic_patients)} patient(s) with multiple classes.")
    print("Problematic patient IDs:", problematic_patients)
    df_zoom = df_zoom[~df_zoom["patient_id"].isin(problematic_patients)].copy()

patient_level = df_zoom[["patient_id", "class"]].drop_duplicates().reset_index(drop=True)
print("\nClean patient-level class distribution:")
print(patient_level["class"].value_counts())

rng = np.random.default_rng(seed)
train_patients, val_patients, test_patients = [], [], []

for cls_name, grp in patient_level.groupby("class"):
    patients = grp["patient_id"].to_numpy().copy()
    rng.shuffle(patients)
    n = len(patients)

    if n < 3:
        raise ValueError(
            f"Class {cls_name} has only {n} patient(s). Need at least 3 for train/val/test coverage."
        )

    # Allocate at least 1 patient to each split for every class.
    n_temp = int(round(n * test_val_size))
    n_temp = max(2, min(n - 1, n_temp))
    n_train = n - n_temp

    n_val = n_temp // 2
    n_test = n_temp - n_val
    if n_val == 0:
        n_val, n_test = 1, n_temp - 1
    if n_test == 0:
        n_test, n_val = 1, n_temp - 1

    train_patients.extend(patients[:n_train].tolist())
    val_patients.extend(patients[n_train:n_train + n_val].tolist())
    test_patients.extend(patients[n_train + n_val:n_train + n_val + n_test].tolist())

train_df = df_zoom[df_zoom["patient_id"].isin(train_patients)].reset_index(drop=True)
val_df = df_zoom[df_zoom["patient_id"].isin(val_patients)].reset_index(drop=True)
test_df = df_zoom[df_zoom["patient_id"].isin(test_patients)].reset_index(drop=True)

# Safety checks: no patient overlap.
assert set(train_df["patient_id"]) & set(val_df["patient_id"]) == set()
assert set(train_df["patient_id"]) & set(test_df["patient_id"]) == set()
assert set(val_df["patient_id"]) & set(test_df["patient_id"]) == set()

print("\nPatient split check:")
print("Train patients:", train_df["patient_id"].nunique())
print("Val patients  :", val_df["patient_id"].nunique())
print("Test patients :", test_df["patient_id"].nunique())

print("\nFinal image-level class distribution:")
print("\nTrain:")
print(train_df["class"].value_counts())
print("\nValidation:")
print(val_df["class"].value_counts())
print("\nTest:")
print(test_df["class"].value_counts())

print("\nFinal sample counts:")
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

print("\nValidation class counts:")
print(val_df["class"].value_counts())

In [ ]:
CLASS_NAMES = ['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(CLASS_NAMES)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

train_df["label"] = train_df["class"].map(class_to_idx)
val_df["label"] = val_df["class"].map(class_to_idx)
test_df["label"] = test_df["class"].map(class_to_idx)

In [ ]:
from tensorflow.keras.applications.densenet import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8
AUTOTUNE = tf.data.AUTOTUNE

def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def apply_preprocess(image, label):
    image = preprocess_input(image)
    return image, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomContrast(0.05),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1)
])

In [ ]:
def make_dataset(df, training=False):
    image_paths = df["file_path"].values
    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                num_parallel_calls=AUTOTUNE)

    ds = ds.map(apply_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()

    if training:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

In [ ]:
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras import applications

def build_pretrained_sequential_model(input_shape=(224, 224, 3), num_classes=8):

    base_model = tf.keras.applications.DenseNet121(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
    )


    model = models.Sequential([
        tf.keras.Input(shape=input_shape),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.LayerNormalization(epsilon=1e-5),
        layers.Dense(256),
        layers.Dropout(0.3),
        layers.Dense(
            num_classes, 
            activation="softmax", 
            kernel_regularizer=tf.keras.regularizers.l2(1e-5)
        )
    ])

    return model, base_model

In [ ]:
from bk_tools import BASE_DIR


model, base_model = build_pretrained_sequential_model(
    input_shape=(224, 224, 3),
    num_classes=NUM_CLASSES
)

MODEL_DIR = BASE_DIR/"models"
MODEL_DIR.mkdir(exist_ok=True)
checkpoint_path = MODEL_DIR / "best_model_densenet.keras"

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10, # Cosine dalgası bazen geç toparlar, sabrı artırdık
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    )
    # ReduceLROnPlateau'ya ARTIK GEREK YOK çünkü CosineDecay bu işi baştan planlıyor.
]


In [ ]:
# === CLASS WEIGHT COMPUTATION (CLEAN VERSION) ===

# 1) Count samples per class
class_counts = train_df["class"].value_counts().to_dict()

total_samples = sum(class_counts.values())
num_classes = len(class_counts)

# 2) Raw inverse frequency
raw_weights = {
    cls: total_samples / (num_classes * count)
    for cls, count in class_counts.items()
}

# 3) Softening (sqrt)
soft_weights = {
    cls: raw_weights[cls] ** 0.5
    for cls in raw_weights
}

# 4) Normalize so minimum weight = 1.0
min_w = min(soft_weights.values())
normalized_weights = {
    cls: w / min_w
    for cls, w in soft_weights.items()
}

# 5) Cap weights to avoid instability
MAX_WEIGHT = 4.0
capped_weights = {
    cls: min(max(w, 1.0), MAX_WEIGHT)
    for cls, w in normalized_weights.items()
}

# 6) Global scaling (reduce aggressiveness)
GLOBAL_SCALE = 0.5
scaled_weights = {
    cls: w * GLOBAL_SCALE
    for cls, w in capped_weights.items()
}

# 7) DC bias control (reduce dominance)
DC_FACTOR = 0.7
scaled_weights["DC"] *= DC_FACTOR

# 8) Convert to label-index map
class_weight_map = {
    class_to_idx[cls]: w
    for cls, w in scaled_weights.items()
}

# === DEBUG PRINT ===
print("Final class weights:")
for cls in CLASS_NAMES:
    print(f"{cls}: {scaled_weights[cls]:.3f}")

print("\nClass weight map (label -> weight):")
print(class_weight_map)

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # stage2 için 5e-6
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
        tf.keras.metrics.SparseCategoricalCrossentropy(name="cross_entropy"),

    ]
)

model.summary()

In [ ]:
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200,
    class_weight=class_weight_map,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # stage2 için 5e-6
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),    
        tf.keras.metrics.SparseCategoricalCrossentropy(name="cross_entropy"),

    ]
)
model.summary()


In [ ]:
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=20,
    epochs=200,
    class_weight=class_weight_map,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    history_dict = history.history

    plt.figure(figsize=(12, 5))
    plt.plot(history_dict["loss"], label="train_loss")
    plt.plot(history_dict["val_loss"], label="val_loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(history_dict["accuracy"], label="train_accuracy")
    plt.plot(history_dict["val_accuracy"], label="val_accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    if "top2_acc" in history_dict and "val_top2_acc" in history_dict:
        plt.figure(figsize=(12, 5))
        plt.plot(history_dict["top2_acc"], label="train_top2_acc")
        plt.plot(history_dict["val_top2_acc"], label="val_top2_acc")
        plt.title("Top-2 Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Top-2 Accuracy")
        plt.legend()
        plt.show()

plot_training_history(history_stage2)

In [ ]:
test_loss, test_acc, test_top2, test_ce = model.evaluate(test_ds, verbose=1)

print(f"Test Loss         : {test_loss:.4f}")
print(f"Test Accuracy     : {test_acc:.4f}")
print(f"Test Top-2 Acc    : {test_top2:.4f}")
print(f"Test CrossEntropy : {test_ce:.4f}")

y_true = test_df["label"].values
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

In [ ]:
from sklearn.metrics import classification_report

CLASS_NAMES = ['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(cm, class_names, title="Confusion Matrix", normalize=False, cmap="Blues"):
    cm = np.asarray(cm)

    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums = np.where(row_sums == 0, 1, row_sums)
        cm_to_plot = cm.astype(np.float64) / row_sums

        annot = np.empty_like(cm, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                annot[i, j] = f"{cm[i, j]}\n({cm_to_plot[i, j] * 100:.1f}%)"
        fmt = ""
    else:
        cm_to_plot = cm
        annot = cm
        fmt = "d"

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm_to_plot,
        annot=annot,
        fmt=fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True,
        linewidths=0.5,
        linecolor="white",
        square=True
    )
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(cm, CLASS_NAMES, title="Confusion Matrix (Counts)", normalize=False)
plot_confusion_matrix(cm, CLASS_NAMES, title="Confusion Matrix (Row-Normalized)", normalize=True)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

# Collect true labels
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

# Predict
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nPrediction Distribution:")
unique, counts = np.unique(y_pred, return_counts=True)
for idx, cnt in zip(unique, counts):
    print(f"{CLASS_NAMES[idx]}: {cnt}")